# Notebook 04 — Exploratory Data Analysis

**Input:** `data/features.csv` — model-ready flat CSV, one row per player per game

**Purpose:** Understand the data before modeling. Identify which features separate
winners from losers, check for multicollinearity, and surface any remaining data quality issues.

**Sections:**
1. Load & profile
2. Win/loss distribution plots per feature
3. Correlation heatmap
4. Feature differentiation summary (effect size ranking)

**Note:** `elo` is retained as reference metadata for context but is excluded from
model features. Plots use the full dataset — no Elo cohort splits (see DEC-009).

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

pd.set_option('display.max_rows', 100)
sns.set_theme(style='whitegrid', palette='muted')

FEATURES_PATH = '../data/features.csv'
PLOT_DIR      = '../data/plots/'

import pathlib
pathlib.Path(PLOT_DIR).mkdir(parents=True, exist_ok=True)

# Columns that are metadata — excluded from feature matrix X
META_COLS = ['result', 'elo', 'civilization_id']

print('Setup OK')

## Section 1 — Load & Profile

In [ ]:
df = pd.read_csv(FEATURES_PATH)
print(f'Shape: {df.shape}')
print(f'Class balance:\n{df["result"].value_counts().to_string()}')
print(f'\nElo range: {df["elo"].min():.0f} – {df["elo"].max():.0f}  (median {df["elo"].median():.0f})')
print(f'\nNull counts per column:')
nulls = df.isnull().sum()
print(nulls[nulls > 0].to_string() if (nulls > 0).any() else '  None')

# Feature matrix (exclude meta)
feature_cols = [c for c in df.columns if c not in META_COLS]
print(f'\nFeature columns ({len(feature_cols)}): {feature_cols}')

## Section 2 — Win/Loss Distribution Plots

For each feature, overlay the distribution for winners (result=1) vs losers (result=0).
Features where the distributions diverge are likely to carry predictive signal.

In [ ]:
wins   = df[df['result'] == 1]
losses = df[df['result'] == 0]

n_cols  = 4
n_rows  = int(np.ceil(len(feature_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    w_vals = wins[col].dropna()
    l_vals = losses[col].dropna()
    ax.hist(l_vals, bins=20, alpha=0.55, color='#e07070', label='Loss', density=True)
    ax.hist(w_vals, bins=20, alpha=0.55, color='#70a8e0', label='Win',  density=True)
    ax.set_title(col, fontsize=9, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(fontsize=8)

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions: Win vs Loss', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
out = f'{PLOT_DIR}distributions.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

## Section 3 — Correlation Heatmap

Pearson correlations across all features. Flag pairs with |r| > 0.7 — high correlation
means redundant signal; consider dropping one before modeling to reduce noise.

In [ ]:
corr = df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(max(14, len(feature_cols) * 0.55),
                                max(12, len(feature_cols) * 0.5)))
mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle only
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.1f', cmap='coolwarm',
    center=0, vmin=-1, vmax=1, linewidths=0.4,
    annot_kws={'size': 6}, ax=ax
)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
out = f'{PLOT_DIR}correlation_heatmap.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

# Flag high-correlation pairs
high_corr = [
    (c1, c2, corr.loc[c1, c2])
    for i, c1 in enumerate(feature_cols)
    for c2 in feature_cols[i+1:]
    if abs(corr.loc[c1, c2]) > 0.7
]
if high_corr:
    print('\nHigh-correlation pairs (|r| > 0.7):')
    for c1, c2, r in sorted(high_corr, key=lambda x: -abs(x[2])):
        print(f'  {c1} ↔ {c2}:  r = {r:.2f}')
else:
    print('\nNo pairs with |r| > 0.7')

## Section 4 — Feature Differentiation Summary

Rank features by how strongly they separate winners from losers using two metrics:
- **Mann-Whitney U p-value**: non-parametric test, works with NaNs and skewed distributions
- **Cohen's d**: standardized effect size — how many standard deviations apart the group means are

Features with low p-value AND high |d| are the strongest coaching candidates.

In [ ]:
results = []
for col in feature_cols:
    w = wins[col].dropna()
    l = losses[col].dropna()
    if len(w) < 5 or len(l) < 5:
        continue
    stat, p = stats.mannwhitneyu(w, l, alternative='two-sided')
    # Cohen's d
    pooled_std = np.sqrt((w.std()**2 + l.std()**2) / 2)
    d = (w.mean() - l.mean()) / pooled_std if pooled_std > 0 else 0.0
    results.append({
        'feature': col,
        'win_mean': w.mean(),
        'loss_mean': l.mean(),
        'p_value': p,
        'cohens_d': d,
        'abs_d': abs(d)
    })

summary = pd.DataFrame(results).sort_values('abs_d', ascending=False)
summary['significant'] = summary['p_value'] < 0.05

print('Feature Differentiation Summary (ranked by effect size):')
print(summary[['feature','win_mean','loss_mean','p_value','cohens_d','significant']]
      .to_string(index=False, float_format=lambda x: f'{x:.3f}'))

out = f'{PLOT_DIR}differentiation_summary.csv'
summary.to_csv(out, index=False)
print(f'\nSaved → {out}')

# Bar chart of top features by effect size
top = summary.head(20)
colors = ['#70a8e0' if d > 0 else '#e07070' for d in top['cohens_d']]
fig, ax = plt.subplots(figsize=(10, 0.45 * len(top) + 1.5))
ax.barh(top['feature'][::-1], top['cohens_d'][::-1], color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel("Cohen's d  (positive = higher in wins)")
ax.set_title('Top Features by Effect Size (Win vs Loss)', fontweight='bold')
plt.tight_layout()
out2 = f'{PLOT_DIR}effect_sizes.png'
plt.savefig(out2, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out2}')